# Sistema simple de recomendación de películas con Python

¿Alguna vez te has preguntado cómo Netflix es capaz de adivinar casi a la perfección la serie o película que vas a ver? Te prometo que no es brujería, es tecnología. Lo que sucede detrás de las recomendacioens que ves cada día en diversas plataformas de streaming como Disney+, Max e inclusive Spotify, hay algoritmos que trabajan con millones de datos para ofrecerte contenido que podría gustarte. 

Uno de los algoritmos empleados en los sistemas de recomendación, se llama Filtrado Colaborativo (FC). ¿Te dejé igual? No te preocupes, vamos a desarrollar esta maraña para que veas cómo funciona esta tecnología, y lo mejor de todo, podrás desarrollar un sistema de recomendaciones simple con Python para que lo presumas con tus amigos.


## 🧰 Primer paso: Importación de librerias

In [91]:
# Librerías asociadas con la carga y lectura de datos 
from surprise import Dataset, Reader
# Librerías asociadas con la creación de modelos de recomendación
from surprise.prediction_algorithms.matrix_factorization import SVD
# Librería asociada a la herramienta para evaluación del modelo
from surprise import accuracy
# Librerías asociadas a la reconversion de datos
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
# Librerías asociadas a la división de datos
from sklearn.model_selection import train_test_split
# Librería asociada a la visualización de datos
import pandas as pd

## 📄 Segundo paso: Carga y pre-procesado de datos 

Para este algoritmo, se usó el conjunto de datos de [MovieLens](https://grouplens.org/datasets/movielens/) en su versión ligera, que contiene aldededor de 100,000 reseñas de 600 usuarios aplicadas a 9,000 películas con 3,600 etiquetas. Los datos fueron descargados en formato zip y fueron descomprimidos en el directorio `/data` de este repositorio. La descripción completa del conjunto de datos se encuentra en el archivo [README](\data\README.txt).

El conjunto de datos incluye 4 archivos: datos de reseñas, datos de etiquetas, datos de películas y datos de enlaces de las películas. Como el objetivo de este algoritmo es la recomendación de películas basadas en las preferencias del usuario, los únicos datos que son de intéres para este proyecto son las reseñas y las películas. 

Por ende, se cargan estos datos a un dataframe de Pandas.

In [92]:
# Carga de los datos de películas y reseñas en un dataframe
movies_df = pd.read_csv('data/movies.csv')
ratings_df = pd.read_csv('data/ratings.csv')

El contenido de las variables `movies_df` y `ratings_df` ahora es el siguiente: 

In [93]:
# Visualización del dataframe de películas
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [94]:
# Visualización del dataframe de reseñas
ratings_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


La visualización de los dataframes anteriores hace más sencilla la compresión de los tipos de datos que existen en el conjunto que se está evaluando, puede observarse que en el primero se muestra información de la película (ID, género y nombre de la película), mientras que en el segundo, las calificaciones que los usuarios han puesto sobre las películas (UserID, MovieID, calificación y estampa de tiempo).

El siguiente paso es realizar una combinación de ambos dataframes, teniendo como foco el ID de la película para realizar la unión, esto con el objetivo de que se sepa qué tipo o género de película se está evaluando.

Lo anterior se logrará uniendo el conjunto de `ratings_df` con un subconjunto de datos de `movies_df` (es un subconjunto porque solo se tomarán los datos provenientes de dos columnas `movieID` y `genres`), emparejando toda la información de ambos sets con el `movieID`.

In [95]:
# Agrupación de dataframes
df = pd.merge(ratings_df, movies_df[['movieId', 'genres']], on='movieId', how='left')

# Visualización de la agrupación
df

,userId,movieId,rating,timestamp,genres
0,1,1,4.0,964982703,Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Comedy|Romance
2,1,6,4.0,964982224,Action|Crime|Thriller
3,1,47,5.0,964983815,Mystery|Thriller
4,1,50,5.0,964982931,Crime|Mystery|Thriller
...,...,...,...,...,...
100831,610,166534,4.0,1493848402,Drama|Horror|Thriller
100832,610,168248,5.0,1493850091,Action|Crime|Thriller
100833,610,168250,5.0,1494273047,Horror
100834,610,168252,5.0,1493846352,Action|Sci-Fi


El siguiente paso, aunque puede verse un poco complejo, es tomar los datos de los usuarios, las películas y los géneros para convertirlos en una matriz de valores númericos ordenados o binarios (0 y 1), es algo que resulta obligatorio para que una  computadora pueda procesarlos de forma fácil en un sistema de recomendación.

Para esto, se utilizarán clases provenientes de la libreria Scikit que ayudan a realizar una codificación de los datos, estas clases son `LabelEncoder` y `MultiLabelBinarizer`, que como se mencionó, ayudan en la conversión de datos categóricos (tales como nombres o etiquetas) en números. Para instanciar estas clases, se definen las variables `user_encoder` y `movie_encoder` usando *LabelEncoder* para convertir los identificadores de usuarios y películas en números, y por otro lado, para los géneros se utilizará la clase *MultiLabelBinarizer* para separar los géneros en múltiples columnas para formar una matriz de valores binarios (0s y 1s)

In [96]:
# Instanciación de los codificadores
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()
genre_encoder = MultiLabelBinarizer()

# Conversión de identificadores a valores numéricos ordenados
# Ej. userId = [1, 2, 3, 4, 5] -> userId = [0, 1, 2, 3, 4]
df['userId'] = user_encoder.fit_transform(df['userId'])
df['movieId'] = movie_encoder.fit_transform(df['movieId'])

# Conversión de géneros a matriz binaria mediante extracción y división de géneros que están separados por '|' 
df = df.join(pd.DataFrame(genre_encoder.fit_transform(df.pop('genres').str.split('|')), columns=genre_encoder.classes_, index=df.index))

# Visualización del pre-procesamiento y agrupamiento de datos
df

,userId,movieId,rating,timestamp,(no genres listed),Action,Adventure,Animation,Children,Comedy,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,4.0,964982703,0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,2,4.0,964981247,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
2,0,5,4.0,964982224,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,43,5.0,964983815,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,0,46,5.0,964982931,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100831,609,9416,4.0,1493848402,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
100832,609,9443,5.0,1493850091,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
100833,609,9444,5.0,1494273047,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
100834,609,9445,5.0,1493846352,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


Como último punto en este paso, se eliminan las categorías vacías

In [97]:
df.drop(columns = "(no genres listed)", inplace = True)

# Visualización del dataframe final
df

,userId,movieId,rating,timestamp,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,4.0,964982703,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,2,4.0,964981247,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
2,0,5,4.0,964982224,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,0,43,5.0,964983815,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,0,46,5.0,964982931,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100831,609,9416,4.0,1493848402,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
100832,609,9443,5.0,1493850091,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
100833,609,9444,5.0,1494273047,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
100834,609,9445,5.0,1493846352,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


## <img align="left" alt="Eyes" width="35px" style="padding-right:10px;" src="https://fonts.gstatic.com/s/e/notoemoji/latest/1f916/512.webp"/>  Tercer paso: Desarrollo del algoritmo de filtrado colaborativo

In [99]:
# División de datos en entrenamiento y prueba con un tamaño de prueba del 20%
train_df, test_df = train_test_split(df, test_size = 0.2)

# Visualización del dataframe de entrenamiento que contiene el 80% de los datos
train_df

,userId,movieId,rating,timestamp,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
49342,317,5150,3.0,1237233422,0,1,1,1,1,0,...,0,0,0,1,0,1,0,0,0,0
18958,121,3623,5.0,1461561706,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
40235,273,5780,3.5,1171410749,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
17607,110,6813,3.5,1516152711,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
81734,516,2658,0.5,1488398787,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68579,446,30,4.0,836961195,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7285,49,6737,2.0,1514239001,1,1,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
24115,166,1075,3.5,1154721626,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,0,0
756,5,383,4.0,845553457,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0


In [100]:
reader = Reader(rating_scale=(0.5, 5))
data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)

trainset = data.build_full_trainset()
trainset

In [102]:
model_svd = SVD()
model_svd.fit(trainset)

predictions_svd = model_svd.test(trainset.build_anti_testset())
accuracy.rmse(predictions_svd)

RMSE: 0.4750


0.4750096969279441

## <img align="left" alt="Eyes" width="35px" style="padding-right:10px;" src="https://fonts.gstatic.com/s/e/notoemoji/latest/1f52e/512.webp"/>  Cuarto paso: Generación de predicciones

In [115]:
def get_n_recommendations(user_id, n=5):
    """
    Obtener las N mejores recomendaciones de películas para un usuario dado.

    Parámetros:
    user_id (int): El ID del usuario para quien se generarán las recomendaciones.
    n (int, opcional): El número de recomendaciones a devolver. Por defecto es 5.

    Retorna:
    list: Una lista de títulos de películas recomendadas para el usuario.
    """
    user_movies = df[df['userId'] == user_id]['movieId'].unique()
    all_movies = df['movieId'].unique()
    movies_to_predict = list(set(all_movies) - set(user_movies))

    user_movie_pairs = [(user_id, movie_id, 0) for movie_id in movies_to_predict]
    predictions_cf = model_svd.test(user_movie_pairs)

    top_recommendations = sorted(predictions_cf, key = lambda x: x.est, reverse= True)[:n]

    top_movieids = [int(pred.iid) for pred in top_recommendations]

    top_movies = movie_encoder.inverse_transform(top_movieids)

    return top_movies

In [118]:
user_id = 222
recommendations = get_n_recommendations(user_id, 30)

top_movieTitles = movies_df[movies_df['movieId'].isin(recommendations)]['title'].tolist()
print(f"Top {len(top_movieTitles)} recomendaciones para el usuario {user_id}:")
for title in top_movieTitles:
    print(title)

Top 30 recomendaciones para el usuario 222:
Heat (1995)
Usual Suspects, The (1995)
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
Godfather, The (1972)
Philadelphia Story, The (1940)
Casablanca (1942)
Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966)
12 Angry Men (1957)
Lawrence of Arabia (1962)
Apocalypse Now (1979)
Ran (1985)
Glory (1989)
Touch of Evil (1958)
Bridge on the River Kwai, The (1957)
Chinatown (1974)
Cool Hand Luke (1967)
This Is Spinal Tap (1984)
Seven Samurai (Shichinin no samurai) (1954)
Dangerous Liaisons (1988)
American History X (1998)
Fight Club (1999)
Guess Who's Coming to Dinner (1967)
Donnie Darko (2001)
Kiss Kiss Bang Bang (2005)
Casino Royale (2006)
Inception (2010)
Dark Knight Rises, The (2012)
Django Unchained (2012)
Spotlight (2015)
Logan (2017)
